# LangChain Framework Fundamentals

In [1]:
import os
from dotenv import load_dotenv
load_dotenv() 
api_key = os.environ.get("GROQ_API_KEY")

In [2]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.3-70b-versatile",api_key=api_key,temperature=0.8)

### Prompt Template

It's a string template that accepts a set of parameters from the user to generate a prompt for a language model.

In [4]:
from langchain_core.prompts import PromptTemplate

In [5]:
template = "Write a short paragraph about {topic}"
prompt_template_name = PromptTemplate(input_variables=['topic'],template=template)
print(prompt_template_name.format(topic="photosynthesis"))

Write a short paragraph about photosynthesis


In [6]:
template = "Write a slogan for {topic}"
prompt_template_name = PromptTemplate(input_variables=['topic'],template=template)
print(prompt_template_name.format(topic="climate change awareness"))

Write a slogan for climate change awareness


In [8]:
template = "Translate the following sentence into {language}: I love you"
prompt_template_name = PromptTemplate(input_variables=['language'],template=template)
print(prompt_template_name.format(language="French"))

Translate the following sentence into French: I love you


### Simple LLM Chain

A chain that formats a prompt and calls an LLM.

In [3]:
from langchain_classic.chains import LLMChain

In [7]:
prompt = PromptTemplate.from_template("Write a slogan for {topic}")
#chain = LLMChain(llm=llm,prompt=prompt)
chain = prompt | llm
response = chain.invoke("climate change awareness")
print(response.content)

"Rise for the Future: Act Now, Thrive Tomorrow"

Alternatively, here are a few more options:

1. "Small Steps Today, A Greener Tomorrow"
2. "Change Your Ways, Save the Planet's Days"
3. "Unite for the Earth, It's Our Only Birth"
4. "Cool the Planet, Not the Future"
5. "Heal the Earth, Heal Ourselves"

Choose the one that resonates with you the most or feel free to modify them to create your own unique slogan.


In [26]:
prompt = PromptTemplate.from_template("Create a product name to sell {product}.Only create one name.")
chain = LLMChain(llm=llm,prompt=prompt)
chain.invoke("rainbow popsicles")

{'product': 'rainbow popsicles', 'text': '"Spectrum Sticks"'}

In [30]:
prompt = PromptTemplate.from_template("Create a product name to sell {product}.Only create one name.")
chain = prompt | llm
response = chain.invoke("rainbow popsicles")
print(response.content)

Sunshine Stix


In [6]:
prompt = PromptTemplate.from_template("Write a short paragraph about {topic}")
chain = prompt | llm
response = chain.invoke("pop music")
print(response.content)

Pop music is a dynamic and ever-evolving genre that has captivated audiences for decades. Characterized by its catchy melodies, upbeat rhythms, and often, memorable lyrics, pop music has become a staple of modern entertainment. With its ability to transcend cultural boundaries and appeal to a wide range of listeners, pop music has given rise to iconic artists such as Michael Jackson, Madonna, and Taylor Swift, who have all left an indelible mark on the industry. From the infectious dance beats of the 1970s and 1980s to the genre-bending sounds of today, pop music continues to innovate and push the boundaries of what is possible, making it one of the most popular and enduring genres of music.


### Sequential Chain

One chain's output is used as another chain's input

In [10]:
from langchain_classic.chains import SequentialChain
from langchain_core.runnables import RunnableSequence

In [11]:
prompt1 = PromptTemplate.from_template("Come up with a restaurant name for {cuisine} cuisine")
prompt2 = PromptTemplate.from_template("Give some menu items for {name} restaurant")
name_chain = prompt1 | llm
menu_chain = prompt2 | llm
chain = RunnableSequence(name_chain, menu_chain)
result = chain.invoke({"cuisine": "Indian"})
print("Menu Items:\n", result.content)

Menu Items:
 Based on the provided Indian restaurant name suggestions, I'll create a sample menu for each of the top 3 name suggestions: 

**1. Tandoori Nights**

* Appetizers: 
  - Vegetable Samosas ($6.99)
  - Chicken Tikka ($8.99)
* Entrees: 
  - Tandoori Chicken ($16.99)
  - Palak Paneer ($15.99)
  - Saag Aloo ($14.99)
* Sides: 
  - Basmati Rice ($3.99)
  - Naan Bread ($4.99)
* Desserts: 
  - Gulab Jamun ($6.99)
  - Kulfi ($5.99)

**2. Spice Route**

* Appetizers: 
  - Papri Chaat ($7.99)
  - Paneer Tikka ($9.99)
* Entrees: 
  - Chicken Korma ($17.99)
  - Lamb Vindaloo ($19.99)
  - Vegetable Biryani ($16.99)
* Sides: 
  - Raita ($4.99)
  - Garlic Naan ($5.99)
* Desserts: 
  - Jalebi ($7.99)
  - Ras Malai ($8.99)

**3. Dhaba**

* Appetizers: 
  - Onion Bhajis ($6.99)
  - Chicken Pakora ($8.99)
* Entrees: 
  - Chicken Curry ($14.99)
  - Mattar Paneer ($15.99)
  - Chana Masala ($13.99)
* Sides: 
  - Jeera Rice ($3.99)
  - Roti ($4.99)
* Desserts: 
  - Barfi ($6.99)
  - Falooda ($7.99)

In [13]:
prompt1 = PromptTemplate.from_template("Come up with one good brand name for a company that sells {product}")
prompt2 = PromptTemplate.from_template("Come up with a single slogan for {company}")
company_chain = prompt1 | llm
slogan_chain = prompt2 | llm
chain = RunnableSequence(company_chain, slogan_chain)
result = chain.invoke({"product": "Athleisure"})
print(result.content)

"Elevate Your Style, Elevate Your Performance"


In [15]:
prompt1 = PromptTemplate.from_template("Come up with one good brand name for a company that sells {product}")
prompt2 = PromptTemplate.from_template("Come up with a single slogan for {company_name}")
company_chain = LLMChain(llm=llm, prompt=prompt1,output_key="company_name")
slogan_chain = LLMChain(llm=llm, prompt=prompt2, output_key="slogan")
sequential_chain = SequentialChain(
    chains=[company_chain, slogan_chain],
    input_variables=["product"],
    output_variables=["company_name", "slogan"],
    verbose=True
)
result = sequential_chain.invoke({"product": "Athleisure"})
print("Company name:", result["company_name"])
print("Slogan:", result["slogan"])



> Entering new SequentialChain chain...

> Finished chain.
Company name: **Elevé**: This name suggests a sense of elevation and upward movement, which is fitting for an athleisure brand that aims to help customers elevate their active lifestyle. It's also easy to remember, pronounce, and has a modern, sleek sound to it. The accent mark (é) adds a touch of sophistication and elegance, which can help the brand stand out in a crowded market.
Slogan: "Elevate Your Everyday"


In [16]:
prompt1 = PromptTemplate.from_template("Come up with one good brand name for a company that sells {product}")
prompt2 = PromptTemplate.from_template("Come up with a list of items {company_name} could sell")
company_chain = LLMChain(llm=llm, prompt=prompt1,output_key="company_name")
items_chain = LLMChain(llm=llm, prompt=prompt2, output_key="items")
sequential_chain = SequentialChain(
    chains=[company_chain, items_chain],
    input_variables=["product"],
    output_variables=["company_name", "items"],
    verbose=True
)
result = sequential_chain.invoke({"product": "Makeup"})
print("Company name:", result["company_name"])
print("Items for sale:", result["items"])



> Entering new SequentialChain chain...

> Finished chain.
Company name: Here's a brand name idea for a makeup company: **Aurora Beauty Co.**

This name suggests the idea of a new beginning or a fresh start, which is often associated with the dawn of a new day (Aurora is the name of the Roman goddess of the dawn). It also implies a sense of radiance and glow, which is fitting for a makeup brand that helps people enhance their natural beauty. The "Co." suffix adds a touch of professionalism and simplicity to the name. Overall, Aurora Beauty Co. is a catchy and memorable name that could work well for a makeup company.
Items for sale: Here's a list of potential products that Aurora Beauty Co. could sell:

1. **Aurora Glow Foundation**: A range of foundation shades that provide a natural, radiant finish.
2. **Dawn to Dusk Eyeshadows**: A collection of eyeshadow palettes with shades that evoke the soft, warm hues of sunrise and sunset.
3. **Lumina Lip Glosses**: A line of lip glosses with

### Conversation Memory

Allows chatbot to remember previous messages

In [22]:
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationChain

#### Conversation Buffer Memory

Conversation Buffer Memory stores the full, unsummarised conversation history as a simple buffer of messages. This helps models maintain context.

In [19]:
memory = ConversationBufferMemory(return_messages=True)

/var/folders/t_/1r0rf0nj71lbvfn8t13yq0zc0000gn/T/ipykernel_58755/3614923015.py:1: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(return_messages=True)


In [26]:
conversation = ConversationChain(llm=llm,memory=memory,verbose=True)
print(conversation.predict(input="Hi, my name is Sana."))
print(conversation.predict(input="What is my name?"))
print(conversation.predict(input="What did I just ask you?"))



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[HumanMessage(content='Hi, my name is Sana.', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello Sana, it's lovely to meet you. I'm an artificial intelligence language model, and I'm excited to chat with you. I've been trained on a massive dataset of text from the internet, which includes a wide range of topics, from science and history to entertainment and culture. I can provide information on everything from the latest scientific discoveries to the plot of your favorite TV show. I'm a large language model, I have been trained on over 1.5 billion parameters, and my knowledge cutoff is currently December 2023, so I'll do my best to provide you with t

#### Conversation Summary Memory

Conversation summary condenses past interactions into concise summaries that the model can reference. This helps maintain context over long conversations.

### Parse Structured Output

In [29]:
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser

In [31]:
parser = JsonOutputParser()
llm_json = ChatGroq(model="llama-3.3-70b-versatile",api_key=api_key,temperature=0)

In [ ]:
prompt = PromptTemplate(
    input_variables=["topic"],
    template="""
Generate a JSON object with the following keys for the topic '{topic}'. Do not add any commentary or anything besides the JSON output:
- summary: short summary
- key_points: list of 3 key points
- difficulty: "easy", "medium", or "hard"

JSON:
"""
)
chain = prompt | llm | parser
result = chain.invoke({"topic": "bills"})
print(result)

{'summary': 'Understanding and managing bills effectively', 'key_points': ['Track and organize bills to avoid late payments', 'Create a budget to allocate funds for bill payments', 'Explore options for reducing or negotiating bills'], 'difficulty': 'medium'}


In [ ]:
prompt = PromptTemplate(
    input_variables=["topic"],
    template="""
Generate a JSON object with the following keys for the topic '{topic}'. Do not add any commentary or anything besides the JSON output:
- name: short summary
- key_points: list of 3 key points
- difficulty: "easy", "medium", or "hard"

JSON:
"""
)
chain = prompt | llm | parser
result = chain.invoke({"topic": "bills"})
print(result)

## Email Writer

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def email_writer(topic,tone,recipient,key_points,sender):
    prompt = ChatPromptTemplate.from_template("""
    Write a {tone} email about {topic}.
    Recipient: {recipient}
    Key points: {key_points}
    Sender's name: {sender}
    """)
    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({
        "topic": topic,
        "tone": tone,
        "recipient": recipient,
        "key_points": key_points,
        "sender": sender})
    print(result)

In [38]:
topic = "deadline extension due to illness"
tone =  "professional"
recipient = "lecturer David"
key_points = "3 additional days required"
sender = "Sana"

email_writer(topic,tone,recipient,key_points,sender)

Subject: Request for Deadline Extension due to Illness

Dear Lecturer David,

I hope this email finds you well. Unfortunately, I have been unwell for the past few days and am still recovering. As a result, I am facing some challenges in completing the assigned task on time.

After assessing my current situation, I am writing to request a deadline extension of three additional days to submit the work. This extra time will allow me to complete the task to the best of my abilities and meet the expected standards.

I apologize for any inconvenience this may cause and appreciate your understanding in this matter. I am confident that with this brief extension, I will be able to submit quality work that meets your expectations.

Please let me know if this is feasible, and I will ensure that the work is submitted as soon as possible. If you require any further information or documentation, please do not hesitate to ask.

Thank you for your consideration, and I look forward to your response.

B

In [39]:
topic = "want to catch up with old friend"
tone =  "friendly"
recipient = "Sarah"
key_points = "schedule a meet up for coffee"
sender = "Sana"

email_writer(topic,tone,recipient,key_points,sender)

Subject: Long Time No Talk - Let's Catch Up!

Dear Sarah,

I hope this email finds you well. It feels like it's been ages since we last spoke, and I've been thinking about you a lot lately. I was just going through some old photos and memories, and it made me realize how much I miss our conversations and laughter.

I'd love to catch up with you and hear about what's been going on in your life. How about we schedule a meet up for coffee? We could grab a cup of coffee and chat about everything that's happened since we last saw each other. I'm really looking forward to hearing about your new adventures and sharing some of my own with you.

Would you be free to meet up sometime this week or next? Let me know a date and time that works for you, and I'll make sure to adjust my schedule accordingly.

I'm really excited about the possibility of catching up and reliving some old times. It's going to be great to see you again and make some new memories.

Looking forward to hearing back from you 

## Text Summariser

In [44]:
def text_summariser(instructions, text):
    prompt = ChatPromptTemplate.from_template("""
    Summarize the following text.
    Custom Instructions: {instructions}
    Text:{text}
    """)
    chain = prompt | llm | StrOutputParser()
    summary = chain.invoke({
        "text": text,
        "instructions": instructions})
    print(summary)

In [45]:
text = """The increase in global temperatures has led to more frequent and severe weather events, posing a 
significant threat to ecosystems and human societies. One of the major impacts of climate change is the rise 
in sea levels, which results from the melting of polar ice caps and glaciers. Coastal areas are particularly 
vulnerable, as they face higher risks of flooding, storm surges, and erosion. Additionally, the warming 
atmosphere can hold more moisture, leading to intense and unpredictable precipitation patterns. This 
variability can cause both severe droughts and devastating floods, affecting agricultural productivity and
water resources.The effects of climate change are widespread, influencing not only the environment but also 
the socio-economic stability of communities. For example, changing weather patterns can disrupt food supply 
chains, increase the prevalence of diseases, and force people to migrate from their homes. To mitigate these 
effects, countries are investing in adaptive infrastructure, developing early warning systems, and
implementing policies to reduce greenhouse gas emissions. The collaboration between governments, 
scientists, and communities is crucial to building resilience against the adverse impacts of climate 
change. By taking proactive measures, societies can better prepare for the challenges posed by a changing
climate and work towards a more sustainable future."""


instructions = "Summarise this as if you are talking to a 10 year old"

text_summariser(instructions, text)

So, you know how sometimes we have really big storms or super hot summers? That's because of something called climate change. It's like the Earth is getting a fever, and it's making really bad things happen.

The ice at the North and South Poles is melting, which means the sea levels are rising. That's a problem for people who live near the beach, because their homes might get flooded. It's also making it harder to grow food, because the weather is getting weirder. Sometimes it's too dry, and sometimes it's too wet.

All of this is affecting not just animals and plants, but also people. Some people might get sick more easily, or have to leave their homes because of the weather. But don't worry, there are things we can do to help. Countries are working together to build special buildings that can withstand big storms, and to create systems that warn us when something bad is coming.

We can all help by trying to reduce the bad things we do to the Earth, like using too much energy or thro

In [46]:
instructions = "Summarise in 3 bullet points"

text_summariser(instructions, text)

Here are three bullet points summarizing the text:
* The increase in global temperatures is leading to more frequent and severe weather events, posing a significant threat to ecosystems and human societies, with rising sea levels and intense precipitation patterns being major concerns.
* Climate change is having widespread effects, including disrupting food supply chains, increasing disease prevalence, and forcing people to migrate, which can impact the socio-economic stability of communities and affect agricultural productivity and water resources.
* To mitigate the effects of climate change, countries are investing in adaptive infrastructure, developing early warning systems, and implementing policies to reduce greenhouse gas emissions, with collaboration between governments, scientists, and communities being crucial to building resilience and working towards a more sustainable future.


## Conversational Chatbot

In [47]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly and helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])
chain = prompt | llm | StrOutputParser()
# store chat history
store = {}

def get_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chatbot = RunnableWithMessageHistory(
    chain,
    get_history,
    input_messages_key="input",
    history_messages_key="history"
)
def chat(message):
    return chatbot.invoke(
        {"input": message},
        config={"configurable": {"session_id": "user1"}}
    )

print(chat("Hi, my name is Sana and I'm from Kerala!"))

/Users/sanasalim/anaconda3/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3577: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Namaste Sana, it's lovely to meet you. Kerala is such a beautiful state in India, known for its stunning backwaters, delicious cuisine, and rich culture. What brings you here today? Is there something I can help you with or would you like to chat about your favorite things from Kerala?


In [48]:
print(chat("Let's discuss pop music. Name some great artists."))

I'd love to discuss pop music with you, Sana. There are so many talented artists out there. Here are a few great ones:

1. **Ariana Grande**: Known for her powerful vocals and catchy songs like "Thank U, Next" and "7 Rings".
2. **Taylor Swift**: A global superstar with a wide range of hits, from "Shake It Off" to "Bad Blood" and "Folklore".
3. **Billie Eilish**: A young and edgy pop sensation with unique sounds like "Bad Guy" and "When the Party's Over".
4. **Justin Bieber**: From his early days as a teen heartthrob to his current status as a pop icon, with hits like "Sorry" and "Love Yourself".
5. **Dua Lipa**: A British pop star with a string of danceable hits, including "Don't Start Now" and "Levitating".
6. **Shawn Mendes**: A Canadian singer-songwriter with a soulful voice and popular songs like "Stitches" and "Senorita" (his collab with Camila Cabello).
7. **Katy Perry**: A veteran pop star with a long list of catchy tunes, from "Firework" to "Roar" and "Dark Horse".
8. **Harry S

In [49]:
print(chat("My favourite artists are Ariana Grande and Olivia Rodrigo"))

Awesome choices, Sana. Both Ariana Grande and Olivia Rodrigo are incredibly talented artists.

Ariana Grande is known for her powerful, wide-ranging voice and her ability to blend pop, R&B, and hip-hop styles. She's had so many hit songs over the years, from "The Way" and "Problem" to "Thank U, Next" and "Positions". Her music often deals with themes of love, empowerment, and self-discovery.

Olivia Rodrigo, on the other hand, is a newer artist who's already making waves in the music industry. Her debut single "Drivers License" was a huge hit, and her album "Sour" received widespread critical acclaim. Her music often has a more introspective, emotional tone, and she's not afraid to tackle tough topics like heartbreak, anxiety, and growing up.

Both Ariana and Olivia are great role models for young women, and their music often resonates with fans who are looking for authentic, relatable songs.

What is it about Ariana and Olivia's music that speaks to you, Sana? Is there a particular so

In [50]:
print(chat("Do you remember where I'm from and what my name is?"))

I do remember, Sana. You're from Kerala, India, and your name is Sana. I'm so glad we're having a lovely conversation about pop music! Kerala is such a beautiful place, by the way. Have you ever listened to any Malayalam music or do you mostly enjoy international pop like Ariana Grande and Olivia Rodrigo?


## Data Extractor

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

class ExtractedData(BaseModel):
    name: str | None = None
    email: str | None = None
    phone: str | None = None

def extract_structured(text):
    structured_llm = llm.with_structured_output(ExtractedData)
    prompt = ChatPromptTemplate.from_template("""
    Extract structured information from the text.
    Only return valid structured data. Do not add commentary or include anything beyond what is asked.
    Text:
    {text}
    """)
    chain = prompt | structured_llm
    return chain.invoke({"text": text})

In [55]:
text = "Hi, my name is Sana!! You can reach me at +9112345678. My email is sana@gmail.com. It's nice to meet you!"
result = extract_structured(text)
print(result.model_dump())

{'name': 'Sana', 'email': 'sana@gmail.com', 'phone': '+9112345678'}
